# 👻 IndianHorrorTapes — Autonomous 8K/4K Shorts Production Engine
### 100+ Real Indian Horror Stories, Urban Legends & Paranormal Investigations

📺 **YouTube Channel:** [IndianHorrorTapes](https://www.youtube.com/channel/UCNRr8A9gVZQZJi1eMKiXIJw)
⏰ **Release Schedule:** **Wednesday & Sunday at 9:00 PM IST** (15:30 UTC)
🎬 **Format:** 9:16 Portrait Ultra-HD Shorts (8K / 4K / 1080p) with studio Hinglish neural narration and atmospheric procedural audio.

---
### 🚀 Quick Start Instructions:
1. Run **Step 1** to install dependencies (FFmpeg, Edge-TTS, Google Client).
2. Run **Step 2** to verify your YouTube credentials or upload your `token.json`.
3. Select any of the **103 Real Indian Horror Stories** in **Step 3**.
4. Run **Step 4** to generate, render, and preview the 9:16 portrait video.
5. Run **Step 5** to schedule or publish directly to YouTube!

In [ ]:
#@title 📦 Step 1: Initialize Environment & Dependencies
!pip install --upgrade pip -q
!pip install yt-dlp edge-tts google-api-python-client google-auth-oauthlib google-auth-httplib2 requests static-ffmpeg huggingface_hub -q
!sudo apt-get update -qq && sudo apt-get install -y -qq ffmpeg fonts-noto-core fonts-noto-cjk fonts-dejavu-core

import os, sys
repo_dir = '/content/colab-YT'
if os.path.exists(repo_dir):
    %cd /content/colab-YT
    !git pull origin main
else:
    print("Working in current directory:", os.getcwd())

print("✅ Environment initialized for IndianHorrorTapes!")

In [ ]:
#@title 🔐 Step 2: Verify IndianHorrorTapes Credentials
import os, json

token_path = 'channels/IndianHorrorTapes/token.json'
secret_path = 'channels/IndianHorrorTapes/youtube_client_secret.json'

if os.path.exists(secret_path):
    print("✅ OAuth Client Secret found:", secret_path)
else:
    print("⚠️ Notice: youtube_client_secret.json missing in channels/IndianHorrorTapes/")

if os.path.exists(token_path):
    print("✅ Active YouTube Token found:", token_path)
else:
    print("⚠️ No token.json found yet in channels/IndianHorrorTapes/.")
    print("If you have INDIANHORRORTAPES_TOKEN_JSON, you can paste it below or upload token.json via the Files tab.")

In [ ]:
#@title 🎬 Step 3: Select Story & Quality { run: "auto" }
#@markdown Choose any story number from 1 to 103 (e.g., 1: Bhangarh, 2: Dumas Beach, 3: Nale Ba, 77: Kuldhara):
story_number = 1 #@param {type:"integer"}
quality = "4k" #@param ["8k", "4k", "1080p"]
publish_mode = "Schedule for Wed/Sun 9 PM IST" #@param ["Schedule for Wed/Sun 9 PM IST", "Publish Immediately", "Render Only (No Upload)"]

# Load and display story metadata
sys.path.insert(0, 'channels/IndianHorrorTapes')
from story_generator import get_or_create_story
story = get_or_create_story(story_number)

print(f"🔥 Selected Story #{story['story_number']}: {story['title']}")
print(f"📍 Location: {story.get('location')}")
print(f"🏷️ Tagline: {story.get('tagline')}")
print(f"🎯 Quality: {quality.upper()} Portrait (9:16)")
print(f"🚀 Mode: {publish_mode}")

In [ ]:
#@title 🎥 Step 4: Render Video & Preview In-Notebook
import os
from IPython.display import HTML, display
from video_producer import produce_horror_short

output_dir = 'channels/IndianHorrorTapes/output'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f"IndianHorrorTapes_Ep{story_number:02d}_{quality}.mp4")

print("🎬 Starting production...")
rendered_path = produce_horror_short(story, output_file, quality=quality)

print(f"\n✅ Rendering Complete: {rendered_path}")
print(f"Size: {os.path.getsize(rendered_path) / (1024*1024):.2f} MB")

# Display inline preview player
display(HTML(f'''
<div style="text-align: center; background: #09050d; padding: 20px; border-radius: 12px; max-width: 420px; margin: auto;">
  <h3 style="color: #f87171;">Preview: {story['title']}</h3>
  <video width="360" height="640" controls autoplay muted style="border-radius: 10px; border: 2px solid #ef4444;">
    <source src="{rendered_path}" type="video/mp4">
    Your browser does not support the video tag.
  </video>
</div>
'''))

In [ ]:
#@title 🚀 Step 5: Upload & Schedule to YouTube
from scheduler import run_schedule

if publish_mode == "Render Only (No Upload)":
    print("Render-only mode selected. Upload skipped.")
else:
    is_immediate = (publish_mode == "Publish Immediately")
    print(f"Initiating scheduler run (Immediate: {is_immediate})...")
    vid_id, url = run_schedule(story_number=story_number, quality=quality, force_immediate=is_immediate)
    if vid_id:
        print(f"\n🎉 Successfully uploaded to IndianHorrorTapes!")
        print(f"🔗 Live URL: {url}")
    else:
        print("\nLocal render preserved. Run authenticate_youtube.py or supply INDIANHORRORTAPES_TOKEN_JSON to upload.")

In [ ]:
#@title 🔄 Step 6: Commit Updated State to GitHub
!git config --local user.email "action@github.com"
!git config --local user.name "IndianHorrorTapes Autonomous Bot"
!git add channels/IndianHorrorTapes/state.json
!git diff --staged --quiet || git commit -m "chore(IndianHorrorTapes): update scheduled story state [skip ci]"
!git push origin main || echo "Push complete or no changes to push."